In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append("..") 

In [ ]:
from processing import load_UCI_dataset,extract_all_features,extract_features_from_dataset, select_f_test, select_mrmr, select_reliefF, save_feature_set

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 625, STEP_SIZE= 625)

In [ ]:
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

In [ ]:
test_window = X_train[0, 0, :]

features = extract_all_features(
    test_window,
    fs=125
)

print("Number of features:", len(features))

print(features)

In [ ]:
df_train = extract_features_from_dataset(
    X_train,
    y_train
)

df_val = extract_features_from_dataset(
    X_val,
    y_val
)

df_test = extract_features_from_dataset(
    X_test,
    y_test
)

In [ ]:
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

## Feature Selection ##

We'll reduce the 57 features to 15 using 3 different methods

In [ ]:
feature_columns = [
    col for col in df_train.columns
    if col not in ["SBP", "DBP"]
]

X_train = df_train[feature_columns]
X_val = df_val[feature_columns]
X_test = df_test[feature_columns]

y_train_sbp = df_train["SBP"]
y_train_dbp = df_train["DBP"]

y_val_sbp = df_val["SBP"]
y_val_dbp = df_val["DBP"]

y_test_sbp = df_test["SBP"]
y_test_dbp = df_test["DBP"]

## F-Test ##

In [ ]:
# SBP
f_sbp_features, f_sbp_results = select_f_test(
    X_train,
    y_train_sbp,
    k=15
)
print("F-test selected features for SBP:")

for i, feature in enumerate(f_sbp_features, 1):
    print(i, feature)

In [ ]:
# DBP
f_dbp_features, f_dbp_results = select_f_test(
    X_train,
    y_train_dbp,
    k=15
)

print("\nF-test selected features for DBP:")

for i, feature in enumerate(f_dbp_features, 1):
    print(i, feature)

## mRMR ##

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_regression
from skrebate import ReliefF


# ============================================================
# mRMR FEATURE SELECTION
# ============================================================

def select_mrmr(
    X,
    y,
    k=15,
    sample_size=50000,
    random_state=42
):
    
    X = X.copy()
    y = np.asarray(y, dtype=np.float64)

    # --------------------------------------------------------
    # 1. Sample training data for feature selection
    # --------------------------------------------------------
    
    rng = np.random.RandomState(random_state)

    n_samples = min(sample_size, len(X))

    sample_indices = rng.choice(
        len(X),
        size=n_samples,
        replace=False
    )

    X_sample = X.iloc[sample_indices]
    y_sample = y[sample_indices]

    print(f"mRMR: using {n_samples} samples for feature selection")

    # Convert to numpy
    X_values = X_sample.values.astype(np.float64)
    y_values = y_sample.astype(np.float64)

    n_features = X_values.shape[1]

    # --------------------------------------------------------
    # 2. Relevance: MI(feature, target)
    # --------------------------------------------------------

    relevance = mutual_info_regression(
        X_values,
        y_values,
        random_state=random_state
    )

    # --------------------------------------------------------
    # 3. MI between every pair of features
    # --------------------------------------------------------

    redundancy = np.zeros(
        (n_features, n_features)
    )

    for i in range(n_features):

        for j in range(i + 1, n_features):

            mi_ij = mutual_info_regression(
                X_values[:, [i]],
                X_values[:, j],
                random_state=random_state
            )[0]

            redundancy[i, j] = mi_ij
            redundancy[j, i] = mi_ij

    # --------------------------------------------------------
    # 4. Greedy mRMR selection
    # --------------------------------------------------------

    selected = []

    # First feature = highest relevance
    first_feature = np.argmax(relevance)

    selected.append(first_feature)

    remaining = set(range(n_features))
    remaining.remove(first_feature)

    while len(selected) < k:

        best_feature = None
        best_score = -np.inf

        for candidate in remaining:

            # Average redundancy with
            # already selected features
            avg_redundancy = np.mean([
                redundancy[candidate, s]
                for s in selected
            ])

            # mRMR score
            score = (
                relevance[candidate]
                - avg_redundancy
            )

            if score > best_score:

                best_score = score
                best_feature = candidate

        selected.append(best_feature)
        remaining.remove(best_feature)

    # --------------------------------------------------------
    # 5. Get selected feature names
    # --------------------------------------------------------

    selected_features = [
        X.columns[i]
        for i in selected
    ]

    # --------------------------------------------------------
    # 6. Results table
    # --------------------------------------------------------

    results = pd.DataFrame({
        "feature": X.columns,
        "relevance_MI": relevance
    })

    results = results.sort_values(
        "relevance_MI",
        ascending=False
    ).reset_index(drop=True)

    return selected_features, results


# ============================================================
# RELIEFF FEATURE SELECTION
# ============================================================

def select_reliefF(
    X,
    y,
    k=15,
    n_neighbors=100,
    sample_size=50000,
    random_state=42
):

    X = X.copy()
    y = np.asarray(y)

    # --------------------------------------------------------
    # 1. Sample training data for feature selection
    # --------------------------------------------------------

    rng = np.random.RandomState(random_state)

    n_samples = min(sample_size, len(X))

    sample_indices = rng.choice(
        len(X),
        size=n_samples,
        replace=False
    )

    X_sample = X.iloc[sample_indices]
    y_sample = y[sample_indices]

    print(f"ReliefF: using {n_samples} samples for feature selection")

    # --------------------------------------------------------
    # 2. ReliefF model
    # --------------------------------------------------------

    model = ReliefF(
        n_neighbors=n_neighbors,
        n_features_to_select=k
    )

    model.fit(
        X_sample.values,
        y_sample
    )

    # --------------------------------------------------------
    # 3. Feature importance scores
    # --------------------------------------------------------

    scores = model.feature_importances_

    # Sort from highest to lowest
    indices = np.argsort(scores)[::-1][:k]

    selected_features = [
        X.columns[i]
        for i in indices
    ]

    # --------------------------------------------------------
    # 4. Results table
    # --------------------------------------------------------

    results = pd.DataFrame({
        "feature": X.columns,
        "ReliefF_score": scores
    })

    results = results.sort_values(
        "ReliefF_score",
        ascending=False
    ).reset_index(drop=True)

    return selected_features, results

In [ ]:
mrmr_sbp_features, mrmr_sbp_results = select_mrmr(
    X_train,
    y_train_sbp,
    k=15
)

print("mRMR selected features for SBP:")

for i, feature in enumerate(
    mrmr_sbp_features,
    1
):
    print(i, feature)

In [ ]:
mrmr_dbp_features, mrmr_dbp_results = select_mrmr(
    X_train,
    y_train_dbp,
    k=15
)

print("\nmRMR selected features for DBP:")

for i, feature in enumerate(
    mrmr_dbp_features,
    1
):
    print(i, feature)

## ReliefF ##

In [ ]:
relief_sbp_features, relief_sbp_results = select_reliefF(
    X_train,
    y_train_sbp,
    k=15
)

print("ReliefF selected features for SBP:")

for i, feature in enumerate(
    relief_sbp_features,
    1
):
    print(i, feature)

In [ ]:
relief_dbp_features, relief_dbp_results = select_reliefF(
    X_train,
    y_train_dbp,
    k=15
)

print("\nReliefF selected features for DBP:")

for i, feature in enumerate(
    relief_dbp_features,
    1
):
    print(i, feature)

In [ ]:
selection_summary = {

    "SBP_F_test": f_sbp_features,

    "SBP_mRMR": mrmr_sbp_features,

    "SBP_ReliefF": relief_sbp_features,

    "DBP_F_test": f_dbp_features,

    "DBP_mRMR": mrmr_dbp_features,

    "DBP_ReliefF": relief_dbp_features
}

In [ ]:
for name, features in selection_summary.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for i, feature in enumerate(features, 1):
        print(f"{i:2d}. {feature}")

## Saving ##

In [ ]:
save_feature_set(
    name="SBP_FTEST",
    selected_features=f_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_MRMR",
    selected_features=mrmr_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_RELIEFF",
    selected_features=relief_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

In [ ]:
save_feature_set(
    name="DBP_FTEST",
    selected_features=f_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_MRMR",
    selected_features=mrmr_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_RELIEFF",
    selected_features=relief_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)